# Roboflow keypoint model v10, run locally (no credits)

Roboflow's download page for `football-field-detection/10` states the model is licensed under
**Apache-2.0** and may be used outside Roboflow (without Roboflow support). This notebook:

1. Downloads its weights once (Colab Secret `ROBOFLOW_API_KEY`), kept in `MyDrive/Playbook/field_model/roboflow_v10/`.
2. Loads them with the open-source `rfdetr` package and scores the same 48 test frames as the hosted
   model (hosted: 97% of points found, 10 px median error), with and without Roboflow's contrast stretching.
3. Records its keypoints on both CVAT clips, for Claude to run the full pipeline.

**Runtime:** any GPU (T4 is enough; about 10 minutes). **Runtime -> Run all**; do not edit cells.

In [ ]:
# Cell 1 — GPU, Drive, settings
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
if not os.path.isdir('/content/drive/MyDrive/Playbook'):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = '/content/drive/MyDrive/Playbook/field_model/roboflow_v10'
os.makedirs(DEST, exist_ok=True)
WEIGHTS = f'{DEST}/weights.pt'
WORKSPACE, PROJECT, VERSION = 'muwafag-hussain-o1ghr', 'football-field-detection-f07vi-it2xv', 10
DATA = '/content/drive/MyDrive/Playbook/field_model/v11/v11_coco'  # same images and split as v10 (downloaded earlier)
FIELD_CONF, KP_CONF = 0.30, 0.50
CLIPS = {'hilal_hazm_B': 'HILAL-HAZM_match_B_up10.mp4', 'hilal_ahli_B': 'HILAL-AHLI_match_B-up8.mp4'}

In [ ]:
# Cell 2 — Our repo + the two CVAT clips, and RF-DETR
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
VID = '/content/videos'; os.makedirs(VID, exist_ok=True)
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
for f in CLIPS.values():
    if not os.path.exists(f'{VID}/{f}'):
        open(f'{VID}/{f}', 'wb').write(subprocess.run(['git', 'show', f'FETCH_HEAD:{f}'], cwd=PB,
                                                      capture_output=True, check=True).stdout)
!pip install -q "rfdetr[train]==1.11.0"
print('ready')

In [ ]:
# Cell 3 — Download the v10 weights once (kept in Drive)
import requests
if not os.path.exists(WEIGHTS):
    from google.colab import userdata
    key = userdata.get('ROBOFLOW_API_KEY')
    url = requests.get(f'https://api.roboflow.com/{WORKSPACE}/{PROJECT}/{VERSION}/ptFile', params={'api_key': key},
                       timeout=60).json()['weightsUrl']
    with requests.get(url, stream=True, timeout=600) as r, open(WEIGHTS + '.tmp', 'wb') as f:
        r.raise_for_status()
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    os.replace(WEIGHTS + '.tmp', WEIGHTS)
print('weights:', WEIGHTS, round(os.path.getsize(WEIGHTS) / 1e6, 1), 'MB')

In [ ]:
# Cell 4 — Load locally and score the 48 test frames (hosted v10: 97% found, 10 px median)
import cv2, numpy as np, pandas as pd, torch
from vision.field_model import load_field_model
from vision.detect import infer_field_keypoints
from tools.keypoint_dataset import keypoint_dataset
SPLIT = '/content/field_split_v10'
if not os.path.exists(f'{SPLIT}/test/_annotations.coco.json'):
    shutil.rmtree(SPLIT, ignore_errors=True)
    keypoint_dataset(DATA, SPLIT, by_match=False)  # Roboflow's own split: the test frames v10 never saw
test = json.load(open(f'{SPLIT}/test/_annotations.coco.json'))
summary = {}
for stretch in (True, False):
    field = load_field_model(WEIGHTS, kind='roboflow_v10', stretch=stretch)
    rows = []
    for im in test['images']:
        img = cv2.imread(f"{SPLIT}/test/{im['file_name']}")
        kp = infer_field_keypoints(field, img, FIELD_CONF)
        pred = {int(l): (xy, c) for l, xy, c in zip(kp.class_id[0], kp.xy[0].tolist(), kp.confidence[0])}
        for a in [a for a in test['annotations'] if a['image_id'] == im['id']]:
            for lab in range(1, 33):
                x, y, v = a['keypoints'][3 * (lab - 1):3 * lab]
                if v > 0:
                    (px, py), c = pred.get(lab, ((np.nan, np.nan), 0.0))
                    rows.append({'label': lab, 'conf': float(c), 'err_px': float(np.hypot(px - x, py - y))})
    t = pd.DataFrame(rows); f = t[t.conf >= KP_CONF]
    # points confidently predicted but not labelled in the frame (should be rare)
    name = 'stretch' if stretch else 'plain'
    summary[name] = {'points': len(t), 'found_share': round(len(f) / len(t), 3),
                     'median_err_px': round(float(f.err_px.median()), 1), 'p90_err_px': round(float(f.err_px.quantile(.9)), 1)}
    print(name, summary[name])
best = max(summary, key=lambda k: (summary[k]['found_share'] > 0.9, -summary[k]['median_err_px']))
print('using:', best)
json.dump({'test': summary, 'best': best}, open(f'{DEST}/v10_eval.json', 'w'), indent=1)

In [ ]:
# Cell 5 — Keypoints on both CVAT clips (plain frames, as the pipeline will give them) + speed
field = load_field_model(WEIGHTS, kind='roboflow_v10', stretch=(best == 'stretch'))
speed = {}
for name, f in CLIPS.items():
    cap = cv2.VideoCapture(f'{VID}/{f}'); out, i, times = [], 0, []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        torch.cuda.synchronize(); t0 = time.time()
        kp = infer_field_keypoints(field, frame, FIELD_CONF)
        torch.cuda.synchronize(); times.append(time.time() - t0)
        out += [{'frame': i, 'label': int(l), 'x': round(float(x), 1), 'y': round(float(y), 1), 'conf': round(float(c), 3)}
                for l, (x, y), c in zip(kp.class_id[0], kp.xy[0].tolist(), kp.confidence[0])]
        i += 1
    cap.release()
    pd.DataFrame(out).to_csv(f'{DEST}/{name}_kp.csv', index=False)
    speed[name] = 1000 * float(np.median(times[5:]))
    print(f'{name}: {i} frames, {speed[name]:.1f} ms/frame')
res = json.load(open(f'{DEST}/v10_eval.json')); res['ms_per_frame'] = speed; res['gpu'] = g.stdout.strip()
json.dump(res, open(f'{DEST}/v10_eval.json', 'w'), indent=1)
print(f'\nDone. Tell Claude the results are in {DEST}.')